# Q18: VRAM for Fine-tuning a 7B LLM
**Topic:** LLM | **Difficulty:** Hard | **Time:** 15 min
**Sheet:** GrindLLM50

---

## Question
How much VRAM is required to fine-tune a 7B LLM?

## Detailed Answer

### Full Fine-tuning (FP16, AdamW)
| Component | Formula | 7B Model |
|-----------|---------|----------|
| Model weights (FP16) | 2 × params | 14 GB |
| Gradients (FP16) | 2 × params | 14 GB |
| Adam states (FP32 m + v) | 8 × params | 56 GB |
| Activations | Depends on batch/seq | 5-20 GB |
| **Total** | | **~90-100 GB** |

**Rule of thumb**: Full fine-tuning ≈ **4× model size** (FP16) minimum, often **6-7×** with Adam.

### Reducing VRAM

**1. LoRA (Low-Rank Adaptation)**
- Freeze base model, train small rank-r matrices: $W' = W + BA$ where $B \in R^{d \times r}$, $A \in R^{r \times d}$
- Typical: r=8-64, ~0.1-1% of params trainable
- **VRAM**: ~16-20 GB for 7B (model weights + small optimizer states)

**2. QLoRA**
- Base model quantized to 4-bit (NF4)
- LoRA adapters in FP16
- **VRAM**: ~6-10 GB for 7B

**3. Gradient Checkpointing**
- Trade compute for memory: recompute activations instead of storing
- Saves ~50-70% activation memory, ~30% slower

**4. Mixed Precision (BF16)**
- Forward/backward in BF16, master weights in FP32
- Reduces memory, faster on modern GPUs

**5. DeepSpeed ZeRO**
- ZeRO Stage 1: Shard optimizer states
- ZeRO Stage 2: + shard gradients
- ZeRO Stage 3: + shard model weights
- ZeRO-Offload: Move to CPU RAM

### Practical Summary
| Method | VRAM (7B) | GPUs Needed |
|--------|----------|------------|
| Full FT (FP16 + AdamW) | ~90 GB | 2× A100 80GB |
| Full FT + ZeRO-3 | ~30 GB/GPU | 2-4× A100 |
| LoRA (FP16) | ~18 GB | 1× A100 |
| QLoRA (4-bit) | ~8 GB | 1× RTX 4090 |
| QLoRA + grad ckpt | ~6 GB | 1× RTX 3090 |

## Key Takeaways
- Full 7B fine-tune: **~90 GB VRAM** (Adam states dominate)
- **QLoRA** brings it to **~8 GB** — fits on consumer GPUs
- Gradient checkpointing saves ~50% activation memory at ~30% speed cost
- Formula: weights(2N) + gradients(2N) + Adam(8N) + activations ≈ 12-14× N bytes